# Crypto - Binance trade files

In [1]:
import numpy as np
import pandas as pd
import time

import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
sns.set_style("whitegrid")

import polars as pl

import gzip

# import datetime

Reading and processing of a long list of monthly csv.gz files is simply too slow. Create parquet files which will read much faster. Additionally adding the processing steps below to file will save extra time. 

In [3]:
%%time

# Read gz files using Polars rather than Pandas because significantly faster.

# symbol = 'BTCUSDT'
symbol = 'ETHUSDT'
# symbol = 'BNBUSDT'
# symbol = 'DOGEUSDT'
# symbol = 'XRPUSDT'
# symbol = 'ADAUSDT'
# symbol = 'AVAXUSDT'
# symbol = 'DOTUSDT'
# symbol = 'LINKUSDT'
# symbol = 'SOLUSDT'
path = "data/crypto/binance/futures/um/monthly/trades/" + symbol + "/"

# dates = ['2024-10', '2024-11']

# dates = ['2020-11', '2020-12',
#          '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06',
#          '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12',
#          '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06',
#          '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12',
#          '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
#          '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12',
#          '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06',
#          '2024-07', '2024-08', '2024-09', '2024-10', '2024-11']

dates = ['2026-04', '2026-05']

headers = ['id', 'price', 'qty', 'quote_qty', 'time', 'is_buyer_maker']

# half_tick = {'BTCUSDT': 0.05, 'ETHUSDT': 0.005}


for d in dates:
    
    print(d)
    
    file = path + symbol + "-trades-" + d + ".csv.gz"
    
    # Some Binance trade files are missing the header row
    # Best solution to handle this is to check if present 
    with gzip.open(file) as f:
        line = f.readline()
        i_d, price, qty, quote_qty, t, is_buyer_maker = line.decode('utf-8').strip().split(',')
       
    t0 = time.perf_counter()
    
    if i_d == 'id': # Header row present
        dl = pl.read_csv(file)
    else:
        print('** No header row **')
        dl = pl.read_csv(file, has_header=False)
        dl.columns = headers
        
    t1 = time.perf_counter()
    print(f"Read csv.gz{t1 - t0: 0.1f} seconds")
    
    df = dl.to_pandas()
    t2 = time.perf_counter()
    print(f"Convert to Pandas{t2 - t1: 0.1f} seconds")
    
    # Aggregate partial fills
    l0 = len(df)
    df = df.groupby('time').agg({'price': 'first', 'qty': 'sum', 'time': 'first', 'is_buyer_maker': 'first'})
    df['time_str'] = pd.to_datetime(df['time'], unit='ms')
    
#     # Need to calculate tick size for each day since it may change dynamically
#     tick_size_df = df[['time_str', 'price']].copy()
#     tick_size_df['tick_size'] = tick_size_df['price'].shift(-1) - tick_size_df['price']
#     tick_size_df.dropna(inplace=True)
#     ## Remove zero price changes and then resample daily retaining minimum price change as tick size
#     tick_size_df = tick_size_df[abs(tick_size_df['tick_size'] > 0)]
#     tick_size_df.index = pd.DatetimeIndex(tick_size_df['time_str'])
#     tick_size_df = tick_size_df.resample("1d").agg({'tick_size': 'min'})
#     tick_size_df['date'] = tick_size_df.index
#     tick_size_df['date'] = tick_size_df['date'].apply(lambda x: x.date()) # Convert datetime to object (string)
#     tick_size_df.reset_index(drop=True, inplace=True)
#     ## df requires a date column to merge
#     df['date'] = df['time_str'].apply(lambda x: x.date()) # This is very slow
#     ## Can now merge both frames to get the tick size for each row
#     df = df.merge(tick_size_df, how='inner', on='date')

#     # Generate mid-price
# #     delta = half_tick[symbol]
#     delta = (df['tick_size'] / 2).to_numpy()
#     price = df['price'].to_numpy()
#     is_buyer_maker = df['is_buyer_maker'].to_numpy()
#     mid = list()
#     for i in range(len(price)):
#         m = price[i]+delta[i] if is_buyer_maker[i] == True else price[i]-delta[i]
#         mid.append(m)    
#     df['mid_est'] = mid

    df['mid_est'] = df['price']
    
    # Calculate responses
#     lags = [1, 10]  # Have never found a use for lag_10
    lags = [1]
#     lags = [1, 5, 10, 20, 50]
    response_labels = list()
    for lag in lags:
        label = 'R_' + str(lag)
        response_labels.append(label)
        df[label] = df['mid_est'].shift(-lag) - df['mid_est']

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True) # Do not need a time index, already have time in a column. 
    
    t3 = time.perf_counter()
    
    l1 = len(df)
    print("Row count reduction:", l0, l1, f"{l0/l1: 0.1f}x")
    print(f"Calculation{t3 - t2: 0.1f} seconds")
    
    parquet_file = path + symbol + "-trades-" + d + ".parquet"
    df.to_parquet(parquet_file, compression='gzip')
    t4 = time.perf_counter()
    print(f"Write Parquet{t4 - t3: 0.1f} seconds")

2026-04
Read csv.gz 35.3 seconds
Convert to Pandas 3.9 seconds
Row count reduction: 160590982 25106415  6.4x
Calculation 16.4 seconds
Write Parquet 30.2 seconds
2026-05
Read csv.gz 29.4 seconds
Convert to Pandas 1.6 seconds
Row count reduction: 118675089 20284789  5.9x
Calculation 13.0 seconds
Write Parquet 24.6 seconds
CPU times: user 2min 36s, sys: 45.9 s, total: 3min 22s
Wall time: 2min 34s


In [3]:
df

,price,qty,time,is_buyer_maker,time_str,mid_est,R_1
0,66937.0,0.029,1772323200008,True,2026-03-01 00:00:00.008,66937.0,0.0
1,66937.0,0.090,1772323200009,True,2026-03-01 00:00:00.009,66937.0,0.1
2,66937.1,0.050,1772323200069,False,2026-03-01 00:00:00.069,66937.1,0.0
3,66937.1,0.100,1772323200092,False,2026-03-01 00:00:00.092,66937.1,0.0
4,66937.1,0.069,1772323200098,False,2026-03-01 00:00:00.098,66937.1,0.0
...,...,...,...,...,...,...,...
31036510,68241.4,0.004,1775001598452,False,2026-03-31 23:59:58.452,68241.4,0.1
31036511,68241.5,0.002,1775001598454,False,2026-03-31 23:59:58.454,68241.5,0.0
31036512,68241.5,0.002,1775001598503,False,2026-03-31 23:59:58.503,68241.5,-0.1
31036513,68241.4,0.021,1775001598797,True,2026-03-31 23:59:58.797,68241.4,0.0


In [5]:
%%time

df = pd.read_parquet(parquet_file)
df.tail()

CPU times: user 2.29 s, sys: 345 ms, total: 2.63 s
Wall time: 556 ms


,price,qty,time,is_buyer_maker,time_str,mid_est,R_1
20845283,62766.0,0.212,1719791999526,True,2024-06-30 23:59:59.526,62766.0,0.1
20845284,62766.1,0.040,1719791999734,False,2024-06-30 23:59:59.734,62766.1,-0.1
20845285,62766.0,0.040,1719791999850,True,2024-06-30 23:59:59.850,62766.0,0.1
20845286,62766.1,0.012,1719791999961,False,2024-06-30 23:59:59.961,62766.1,-0.1
20845287,62766.0,0.003,1719791999985,True,2024-06-30 23:59:59.985,62766.0,0.0


Concatenate monthly Parquet files.

In [2]:
%%time

symbol = 'BTCUSDT'
path = "data/crypto/binance/futures/um/monthly/trades/" + symbol + "/"

dates = ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
         '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12']

# dates = ['2023-01', '2023-02']

out_file = path + symbol + "-trades-2023.parquet"

df_list = list()
for d in dates:
    in_file = path + symbol + "-trades-" + d + ".parquet"
    df_list.append(pd.read_parquet(in_file))
df_large = pd.concat(df_list)
df_large.to_parquet(out_file, compression='gzip')

CPU times: user 10min 20s, sys: 42.8 s, total: 11min 3s
Wall time: 10min 39s
